## California Wildfire Analysis, Impacts and Risk, Geographic Information System (GIS)

### Project GIS Summary

The GIS tasks of this project carry out a workflow for analyzing wildfire conditions across California. The process begins by preparing statewide fire perimeter records, fire hazard zone attributes, and monthly temporal weather station data, organizing these inputs into new geodatabases and cleaning fields to remove gaps and standardize numeric values. After building the refined base layers, the workflow creates dedicated machine learning geodatabases for both Random Forest (RF) and Neural Network (NN) modeling. Spatial joins are performed so each monthly weather observation is matched with nearby fire activity and associated hazard information. A fire/no-fire training label is generated, and the resulting joined feature classes provide the predictor variables derived from the temporal weather fields. The RF model is trained, tested, and evaluated using accuracy, precision, recall, F1-score, ROC curves, and confusion matrices, and the final outputs—including tables and test-set predictions—are written back into the geodatabase. This completes an end-to-end integration of fire hazard data, temporal climate variables, spatial processing, and machine learning within ArcGIS Pro 3.5.


### 1. Preparing Base Fire, Weather, Hazard, and Temporal Datasets (GIS Preprocessing)

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Running California wildfire project workflow using ArcGIS Pro 3.5.
#   Describing overall project configuration and base data preparation.
#   Defining folders, geodatabases, and environment settings.
#   Copying raw fire perimeter and weather station data into project FGDBs.
#   Cleaning important fields and keeping only attributes needed later.
# ---------------------------------------------------------------------------

import arcpy
import os

# Setting environment
arcpy.env.overwriteOutput = True

# Setting main project folder
project_root = r"C:\AH_Files\CalFire_NN_Project"

# Defining source geodatabase with original CAL FIRE and CIMIS data
src_gdb = os.path.join(project_root, r"02_data_work\CAL_FIRE_WorkSpace_A__GIS_DataSets.gdb")

# Defining working geodatabases for cleaned base data
gdb_simple = os.path.join(project_root, "10-SimplePredictions.gdb")
gdb_nn_base = os.path.join(project_root, "11-NN_Predictions.gdb")

# Creating working FGDBs if they do not exist
for gdb_path in [gdb_simple, gdb_nn_base]:
    if not arcpy.Exists(gdb_path):
        arcpy.management.CreateFileGDB(
            os.path.dirname(gdb_path),
            os.path.basename(gdb_path)
        )
        print(f"Creating FGDB: {gdb_path}")
    else:
        print(f"Finding existing FGDB: {gdb_path}")

# Defining source feature classes
src_fire_fc = os.path.join(src_gdb, "CalFire_Master_DataSet_C_Final_for_SpatialAnalysis")
src_weather_fc = os.path.join(src_gdb, "monthly_longlat_Temporal_Data")

# Defining cleaned output feature classes
fire_clean_fc = os.path.join(gdb_simple, "FirePerimeters_Clean")
weather_clean_fc = os.path.join(gdb_simple, "WeatherMonthly_Clean")

# Listing essential fire fields (editing list if schema differs)
fire_keep_fields = [
    "OBJECTID",
    "Shape",
    "IncidentName",
    "FirePerime_GISAcres",
    "FireHazard_FhszDescription",
    "AlarmDate",
    "ControlDate",
    "UnitID",
    "County"
]

# Copying fire perimeters and dropping unwanted fields
if not arcpy.Exists(fire_clean_fc):
    print("Copying fire perimeters into working FGDB")
    arcpy.management.CopyFeatures(src_fire_fc, fire_clean_fc)

    # Keeping only essential fields
    existing_fields = [f.name for f in arcpy.ListFields(fire_clean_fc)]
    drop_fields = [
        f for f in existing_fields
        if f not in fire_keep_fields and f not in ("Shape_Length", "Shape_Area")
    ]
    if drop_fields:
        print("Dropping unnecessary fire fields:")
        print(drop_fields)
        arcpy.management.DeleteField(fire_clean_fc, drop_fields)
else:
    print("Finding existing cleaned fire FC:", fire_clean_fc)

# Listing essential weather fields (editing list to match actual schema)
weather_keep_fields = [
    "OBJECTID",
    "Shape",
    "StationID",
    "MonthYear",
    "TotalETo",
    "TotalPrecip",
    "AvgVapPres",
    "AvgMaxAirTemp",
    "AvgMinAirTemp",
    "AvgAirTemp",
    "AvgMaxRelHum",
    "AvgMinRelHum",
    "AvgDewPoint",
    "AvgWindSpeed",
    "AvgSoilTemp",
    "Longitude",
    "Latitude"
]

# Copying weather points and dropping unwanted fields
if not arcpy.Exists(weather_clean_fc):
    print("Copying weather station monthly data into working FGDB")
    arcpy.management.CopyFeatures(src_weather_fc, weather_clean_fc)

    existing_fields = [f.name for f in arcpy.ListFields(weather_clean_fc)]
    drop_fields = [
        f for f in existing_fields
        if f not in weather_keep_fields and f not in ("Shape_Length", "Shape_Area")
    ]
    if drop_fields:
        print("Dropping unnecessary weather fields:")
        print(drop_fields)
        arcpy.management.DeleteField(weather_clean_fc, drop_fields)
else:
    print("Finding existing cleaned weather FC:", weather_clean_fc)


# Cleaning numeric fields by replacing None or NaN with zero
def clean_numeric_fields(fc, field_names):
    """Using UpdateCursor for replacing null numeric values with zero"""
    existing = [f.name for f in arcpy.ListFields(fc)]
    target_fields = [f for f in field_names if f in existing]
    if not target_fields:
        print("Skipping numeric cleaning for", fc, "- no matching fields")
        return

    print(f"Cleaning numeric fields in {fc}: {target_fields}")
    with arcpy.da.UpdateCursor(fc, target_fields) as cursor:
        for row in cursor:
            changed = False
            new_row = list(row)
            for i, val in enumerate(row):
                if val is None:
                    new_row[i] = 0
                    changed = True
            if changed:
                cursor.updateRow(new_row)


# Cleaning fire numeric field (area)
clean_numeric_fields(fire_clean_fc, ["FirePerime_GISAcres"])

# Cleaning weather numeric predictors
clean_numeric_fields(
    weather_clean_fc,
    [
        "TotalETo",
        "TotalPrecip",
        "AvgVapPres",
        "AvgMaxAirTemp",
        "AvgMinAirTemp",
        "AvgAirTemp",
        "AvgMaxRelHum",
        "AvgMinRelHum",
        "AvgDewPoint",
        "AvgWindSpeed",
        "AvgSoilTemp"
    ]
)

# Cleaning fire hazard descriptive field (replacing null or blank with 'Unknown')
hazard_field = "FireHazard_FhszDescription"
if hazard_field in [f.name for f in arcpy.ListFields(fire_clean_fc)]:
    print("Cleaning fire hazard description values")
    with arcpy.da.UpdateCursor(fire_clean_fc, [hazard_field]) as cursor:
        for (val,) in cursor:
            if val is None or str(val).strip() == "":
                cursor.updateRow(("Unknown",))

print("Finishing base data preparation for fire perimeters and weather stations")


### 2. Building Machine Learning Feature Classes for Random Forest (RF) and Neural Network (NN)

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Running California wildfire project workflow using ArcGIS Pro 3.5.
#   Preparing machine learning master datasets by combining fire and weather.
#   Creating dedicated ML geodatabases for Random Forest and Neural Network.
#   Performing spatial joins between weather points and fire perimeters.
#   Creating fire/no-fire labels and keeping temporal predictor columns.
# ---------------------------------------------------------------------------

import arcpy
import os

# Setting environment
arcpy.env.overwriteOutput = True

# Setting project paths
project_root = r"C:\AH_Files\CalFire_NN_Project"
gdb_simple = os.path.join(project_root, "10-SimplePredictions.gdb")

# Defining cleaned base feature classes from Cell 1
fire_clean_fc = os.path.join(gdb_simple, "FirePerimeters_Clean")
weather_clean_fc = os.path.join(gdb_simple, "WeatherMonthly_Clean")

# Defining ML target geodatabases
rf_gdb = os.path.join(project_root, "14_RF_ML_90_target_DataSet.gdb")
nn_gdb = os.path.join(project_root, "13_NN_ML_90_target_DataSet.gdb")

# Creating ML FGDBs if they do not exist
for gdb_path in [rf_gdb, nn_gdb]:
    if not arcpy.Exists(gdb_path):
        arcpy.management.CreateFileGDB(
            os.path.dirname(gdb_path),
            os.path.basename(gdb_path)
        )
        print(f"Creating ML FGDB: {gdb_path}")
    else:
        print(f"Finding existing ML FGDB: {gdb_path}")

# Defining target feature classes in ML FGDBs
rf_fire_fc = os.path.join(rf_gdb, "FirePerimeters_RFAcc")
rf_weather_fc = os.path.join(rf_gdb, "WeatherStations_RFAcc")
rf_join_fc = os.path.join(rf_gdb, "WeatherWithFire_RFAcc")

nn_fire_fc = os.path.join(nn_gdb, "FirePerimeters_NNAcc")
nn_weather_fc = os.path.join(nn_gdb, "WeatherStations_NNAcc")
nn_join_fc = os.path.join(nn_gdb, "WeatherWithFire_NNAcc")

# Copying cleaned base layers into ML FGDBs (using same geometry twice for independence)
if not arcpy.Exists(rf_fire_fc):
    print("Copying fire perimeters into RF ML FGDB")
    arcpy.management.CopyFeatures(fire_clean_fc, rf_fire_fc)
if not arcpy.Exists(nn_fire_fc):
    print("Copying fire perimeters into NN ML FGDB")
    arcpy.management.CopyFeatures(fire_clean_fc, nn_fire_fc)

if not arcpy.Exists(rf_weather_fc):
    print("Copying weather data into RF ML FGDB")
    arcpy.management.CopyFeatures(weather_clean_fc, rf_weather_fc)
if not arcpy.Exists(nn_weather_fc):
    print("Copying weather data into NN ML FGDB")
    arcpy.management.CopyFeatures(weather_clean_fc, nn_weather_fc)

# Performing spatial join for RF dataset
print("Performing spatial join for RF training dataset")
if arcpy.Exists(rf_join_fc):
    arcpy.management.Delete(rf_join_fc)

arcpy.analysis.SpatialJoin(
    target_features=rf_weather_fc,
    join_features=rf_fire_fc,
    out_feature_class=rf_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)

# Performing spatial join for NN dataset (mirroring RF workflow)
print("Performing spatial join for NN training dataset")
if arcpy.Exists(nn_join_fc):
    arcpy.management.Delete(nn_join_fc)

arcpy.analysis.SpatialJoin(
    target_features=nn_weather_fc,
    join_features=nn_fire_fc,
    out_feature_class=nn_join_fc,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)

# Creating label field (1 = intersects fire, 0 = no fire) based on Join_Count
def add_label_field(join_fc, label_field="label"):
    """Adding fire/no-fire label field to joined training FC"""
    existing_fields = [f.name for f in arcpy.ListFields(join_fc)]
    if label_field not in existing_fields:
        print(f"Adding label field '{label_field}' to {join_fc}")
        arcpy.management.AddField(join_fc, label_field, "SHORT")
    else:
        print(f"Finding existing label field '{label_field}' in {join_fc}")

    join_count_field = "Join_Count"
    if join_count_field not in existing_fields:
        raise RuntimeError(f"Expecting Join_Count field in {join_fc}")

    print("Populating label values from Join_Count")
    with arcpy.da.UpdateCursor(join_fc, [join_count_field, label_field]) as cursor:
        for join_count, label in cursor:
            new_label = 1 if join_count and join_count > 0 else 0
            if label != new_label:
                cursor.updateRow((join_count, new_label))


# Adding labels for both RF and NN training feature classes
add_label_field(rf_join_fc, "label")
add_label_field(nn_join_fc, "label")

print("Finishing ML master dataset preparation for RF and NN training")


### 3. Random Forest (RF) Model Training, Testing, and Accuracy Evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Running California wildfire project workflow using ArcGIS Pro 3.5.
#   Building Random Forest classification model for wildfire occurrence.
#   Extracting predictor variables and labels from RF training feature class.
#   Splitting dataset into training and testing subsets.
#   Training Random Forest with class imbalance handling.
#   Computing accuracy, precision, recall, F1, ROC, PR, and confusion matrix.
# ---------------------------------------------------------------------------

import arcpy
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve
)

# Setting environment
arcpy.env.overwriteOutput = True

# Setting project paths
project_root = r"C:\AH_Files\CalFire_NN_Project"
rf_gdb = os.path.join(project_root, "14_RF_ML_90_target_DataSet.gdb")
rf_join_fc = os.path.join(rf_gdb, "WeatherWithFire_RFAcc")

# Defining predictor and label fields (editing names to match actual schema)
predictor_fields = [
    "TotalETo",
    "TotalPrecip",
    "AvgVapPres",
    "AvgMaxAirTemp",
    "AvgMinAirTemp",
    "AvgAirTemp",
    "AvgMaxRelHum",
    "AvgMinRelHum",
    "AvgDewPoint",
    "AvgWindSpeed",
    "AvgSoilTemp"
]
label_field = "label"

# Checking field existence
existing_fields = [f.name for f in arcpy.ListFields(rf_join_fc)]
for f in predictor_fields + [label_field]:
    if f not in existing_fields:
        raise RuntimeError(f"Expecting field '{f}' in {rf_join_fc}")

# Extracting data into arrays
print("Extracting RF training data from feature class")
field_list = ["OID@"] + predictor_fields + [label_field]
oids = []
X_list = []
y_list = []

with arcpy.da.SearchCursor(rf_join_fc, field_list) as cursor:
    for row in cursor:
        oids.append(row[0])
        X_list.append(row[1:-1])
        y_list.append(row[-1])

X = np.array(X_list, dtype=float)
y = np.array(y_list, dtype=int)
oids = np.array(oids, dtype=int)

print("Finding positive and negative label counts:")
unique, counts = np.unique(y, return_counts=True)
print(dict(zip(unique, counts)))

# Splitting into train and test sets
print("Splitting dataset into train and test subsets")
X_train, X_test, y_train, y_test, oid_train, oid_test = train_test_split(
    X,
    y,
    oids,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Training Random Forest with class_weight 'balanced'
print("Training Random Forest model")
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
    oob_score=True
)
rf_model.fit(X_train, y_train)

# Predicting labels and probabilities
print("Predicting RF test set labels and probabilities")
y_prob = rf_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Computing metrics
print("Computing RF performance metrics")
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred,
    average="binary",
    zero_division=0
)

cm = confusion_matrix(y_test, y_pred)
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_test, y_prob)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")


### 4. Exporting Random Forest (RF) Results to Geodatabase (Statistics, Curves, Predictions)

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Running California wildfire project workflow using ArcGIS Pro 3.5.
#   Writing summary tables and test prediction outputs back into RF ML FGDB.
#   Storing RF model metrics, ROC and PR curves, and confusion matrix.
#   Creating feature class of RF test predictions with probabilities.
# ---------------------------------------------------------------------------

# Preparing tables for writing to FGDB
print("Preparing RF stats tables for geodatabase export")

# Creating stats summary table
stats_array = np.array(
    [
        ("Accuracy", float(accuracy)),
        ("Precision", float(precision)),
        ("Recall", float(recall)),
        ("F1-Score", float(f1)),
        ("ROC_AUC", float(roc_auc))
    ],
    dtype=[("Metric", "U32"), ("Value", "f8")]
)

# Creating confusion matrix table
cm_rows = []
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        cm_rows.append((int(i), int(j), int(cm[i, j])))
cm_array = np.array(
    cm_rows,
    dtype=[("TrueLabel", "i4"), ("PredLabel", "i4"), ("Count", "i4")]
)

# Creating ROC curve table
roc_array = np.array(
    list(zip(fpr, tpr, roc_thresholds)),
    dtype=[("FPR", "f8"), ("TPR", "f8"), ("Threshold", "f8")]
)

# Creating PR curve table
pr_array = np.array(
    list(zip(prec_curve, rec_curve)),
    dtype=[("Precision", "f8"), ("Recall", "f8")]
)

# Defining output table paths
rf_stats_tbl = os.path.join(rf_gdb, "RF_ModelStats_MoreAcc")
rf_cm_tbl = os.path.join(rf_gdb, "RF_ConfMatrix_MoreAcc")
rf_roc_tbl = os.path.join(rf_gdb, "RF_ROC_MoreAcc")
rf_pr_tbl = os.path.join(rf_gdb, "RF_PR_MoreAcc")

for t in [rf_stats_tbl, rf_cm_tbl, rf_roc_tbl, rf_pr_tbl]:
    if arcpy.Exists(t):
        arcpy.management.Delete(t)

arcpy.da.NumPyArrayToTable(stats_array, rf_stats_tbl)
arcpy.da.NumPyArrayToTable(cm_array, rf_cm_tbl)
arcpy.da.NumPyArrayToTable(roc_array, rf_roc_tbl)
arcpy.da.NumPyArrayToTable(pr_array, rf_pr_tbl)

print("Writing RF test predictions back into new feature class")

# Creating feature class with test predictions
rf_test_fc = os.path.join(rf_gdb, "RF_TestPredictions_MoreAcc")
if arcpy.Exists(rf_test_fc):
    arcpy.management.Delete(rf_test_fc)

# Copying subset of features used in testing
oid_field_name = "OBJECTID"
arcpy.management.MakeFeatureLayer(rf_join_fc, "rf_join_lyr")
arcpy.management.SelectLayerByAttribute(
    "rf_join_lyr",
    "NEW_SELECTION",
    f"{oid_field_name} IN ({','.join(str(o) for o in oid_test.tolist())})"
)
arcpy.management.CopyFeatures("rf_join_lyr", rf_test_fc)
arcpy.management.Delete("rf_join_lyr")

# Adding prediction fields
for fld_name, fld_type in [("RF_Prob", "DOUBLE"), ("RF_Pred", "SHORT")]:
    if fld_name not in [f.name for f in arcpy.ListFields(rf_test_fc)]:
        arcpy.management.AddField(rf_test_fc, fld_name, fld_type)

# Creating lookup from OID to prediction
prob_lookup = {int(oid): float(prob) for oid, prob in zip(oid_test, y_prob)}
pred_lookup = {int(oid): int(pred) for oid, pred in zip(oid_test, y_pred)}

with arcpy.da.UpdateCursor(rf_test_fc, [oid_field_name, "RF_Prob", "RF_Pred"]) as cursor:
    for oid, prob_val, pred_val in cursor:
        new_prob = prob_lookup.get(int(oid), None)
        new_pred = pred_lookup.get(int(oid), None)
        if new_prob is not None and new_pred is not None:
            cursor.updateRow((oid, new_prob, new_pred))

print("Finishing Random Forest training and result export")


### 5. Neural Network (NN) Model Training, Testing, and Metrics Extraction

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Running Neural Network (NN) workflow for California wildfire conditions.
#   Extracting predictor variables and labels from the NN training feature class.
#   Splitting dataset into training and testing subsets.
#   Training the NN classifier and computing performance metrics.
# ---------------------------------------------------------------------------

import arcpy
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve
)

# Setting environment
arcpy.env.overwriteOutput = True

# Setting project paths
project_root = r"C:\AH_Files\CalFire_NN_Project"
nn_gdb = os.path.join(project_root, "13_NN_ML_90_target_DataSet.gdb")
nn_join_fc = os.path.join(nn_gdb, "WeatherWithFire_NNAcc")

# Defining predictor and label fields
predictor_fields_nn = [
    "TotalETo",
    "TotalPrecip",
    "AvgVapPres",
    "AvgMaxAirTemp",
    "AvgMinAirTemp",
    "AvgAirTemp",
    "AvgMaxRelHum",
    "AvgMinRelHum",
    "AvgDewPoint",
    "AvgWindSpeed",
    "AvgSoilTemp"
]
label_field_nn = "label"

# Checking field existence
existing_fields_nn = [f.name for f in arcpy.ListFields(nn_join_fc)]
for f in predictor_fields_nn + [label_field_nn]:
    if f not in existing_fields_nn:
        raise RuntimeError(f"Expecting field '{f}' in {nn_join_fc}")

# Extracting data into arrays
print("Extracting NN training data from feature class")
field_list_nn = ["OID@"] + predictor_fields_nn + [label_field_nn]
nn_oids = []
nn_X_list = []
nn_y_list = []

with arcpy.da.SearchCursor(nn_join_fc, field_list_nn) as cursor:
    for row in cursor:
        nn_oids.append(row[0])
        nn_X_list.append(row[1:-1])
        nn_y_list.append(row[-1])

nn_X = np.array(nn_X_list, dtype=float)
nn_y = np.array(nn_y_list, dtype=int)
nn_oids = np.array(nn_oids, dtype=int)

print("Finding positive and negative label counts in NN dataset:")
nn_unique, nn_counts = np.unique(nn_y, return_counts=True)
print(dict(zip(nn_unique, nn_counts)))

# Splitting into train and test sets
print("Splitting NN dataset into train and test subsets")
nn_X_train, nn_X_test, nn_y_train, nn_y_test, nn_oid_train, nn_oid_test = train_test_split(
    nn_X,
    nn_y,
    nn_oids,
    test_size=0.3,
    random_state=42,
    stratify=nn_y
)

# Training Neural Network model
print("Training Neural Network (MLPClassifier) model")
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)
nn_model.fit(nn_X_train, nn_y_train)

# Predicting labels and probabilities
print("Predicting NN test set labels and probabilities")
nn_y_prob = nn_model.predict_proba(nn_X_test)[:, 1]
nn_y_pred = (nn_y_prob >= 0.5).astype(int)

# Computing metrics
print("Computing NN performance metrics")
nn_accuracy = accuracy_score(nn_y_test, nn_y_pred)
nn_precision, nn_recall, nn_f1, _ = precision_recall_fscore_support(
    nn_y_test,
    nn_y_pred,
    average="binary",
    zero_division=0
)

nn_cm = confusion_matrix(nn_y_test, nn_y_pred)
nn_fpr, nn_tpr, nn_roc_thresholds = roc_curve(nn_y_test, nn_y_prob)
nn_roc_auc = auc(nn_fpr, nn_tpr)

nn_prec_curve, nn_rec_curve, nn_pr_thresholds = precision_recall_curve(nn_y_test, nn_y_prob)

print(f"NN Accuracy:  {nn_accuracy:.4f}")
print(f"NN Precision: {nn_precision:.4f}")
print(f"NN Recall:    {nn_recall:.4f}")
print(f"NN F1-score:  {nn_f1:.4f}")
print(f"NN ROC AUC:   {nn_roc_auc:.4f}")


### 6. Exporting Neural Network (NN) metrics, curves, and test predictions to geodatabase

In [ ]:
# ---------------------------------------------------------------------------
# Purpose:
#   Writing Neural Network (NN) results back into NN geodatabase.
#   Creating tables for metrics, confusion matrix, ROC, and PR curves.
#   Building feature class with NN test predictions and probabilities.
# ---------------------------------------------------------------------------

import numpy as np
import arcpy
import os

# Setting environment
arcpy.env.overwriteOutput = True

# Setting project paths
project_root = r"C:\AH_Files\CalFire_NN_Project"
nn_gdb = os.path.join(project_root, "13_NN_ML_90_target_DataSet.gdb")
nn_join_fc = os.path.join(nn_gdb, "WeatherWithFire_NNAcc")

print("Preparing NN stats tables for geodatabase export")

# Creating stats summary table
nn_stats_array = np.array(
    [
        ("Accuracy", float(nn_accuracy)),
        ("Precision", float(nn_precision)),
        ("Recall", float(nn_recall)),
        ("F1-Score", float(nn_f1)),
        ("ROC_AUC", float(nn_roc_auc))
    ],
    dtype=[("Metric", "U32"), ("Value", "f8")]
)

# Creating confusion matrix table
nn_cm_rows = []
for i in range(nn_cm.shape[0]):
    for j in range(nn_cm.shape[1]):
        nn_cm_rows.append((int(i), int(j), int(nn_cm[i, j])))
nn_cm_array = np.array(
    nn_cm_rows,
    dtype=[("TrueLabel", "i4"), ("PredLabel", "i4"), ("Count", "i4")]
)

# Creating ROC curve table
nn_roc_array = np.array(
    list(zip(nn_fpr, nn_tpr, nn_roc_thresholds)),
    dtype=[("FPR", "f8"), ("TPR", "f8"), ("Threshold", "f8")]
)

# Creating PR curve table
nn_pr_array = np.array(
    list(zip(nn_prec_curve, nn_rec_curve)),
    dtype=[("Precision", "f8"), ("Recall", "f8")]
)

# Defining output table paths
nn_stats_tbl = os.path.join(nn_gdb, "NN_ModelStats")
nn_cm_tbl = os.path.join(nn_gdb, "NN_ConfMatrix")
nn_roc_tbl = os.path.join(nn_gdb, "NN_ROC")
nn_pr_tbl = os.path.join(nn_gdb, "NN_PR")

# Deleting existing tables if present
for t in [nn_stats_tbl, nn_cm_tbl, nn_roc_tbl, nn_pr_tbl]:
    if arcpy.Exists(t):
        arcpy.management.Delete(t)

# Writing tables
arcpy.da.NumPyArrayToTable(nn_stats_array, nn_stats_tbl)
arcpy.da.NumPyArrayToTable(nn_cm_array, nn_cm_tbl)
arcpy.da.NumPyArrayToTable(nn_roc_array, nn_roc_tbl)
arcpy.da.NumPyArrayToTable(nn_pr_array, nn_pr_tbl)

print("Writing NN test predictions back into new feature class")

# Creating feature class with NN test predictions
nn_test_fc = os.path.join(nn_gdb, "NN_TestPredictions")
if arcpy.Exists(nn_test_fc):
    arcpy.management.Delete(nn_test_fc)

# Copying subset of features used in NN testing
nn_oid_field_name = "OBJECTID"
arcpy.management.MakeFeatureLayer(nn_join_fc, "nn_join_lyr")
arcpy.management.SelectLayerByAttribute(
    "nn_join_lyr",
    "NEW_SELECTION",
    f"{nn_oid_field_name} IN ({','.join(str(o) for o in nn_oid_test.tolist())})"
)
arcpy.management.CopyFeatures("nn_join_lyr", nn_test_fc)
arcpy.management.Delete("nn_join_lyr")

# Adding prediction fields
for fld_name, fld_type in [("NN_Prob", "DOUBLE"), ("NN_Pred", "SHORT")]:
    if fld_name not in [f.name for f in arcpy.ListFields(nn_test_fc)]:
        arcpy.management.AddField(nn_test_fc, fld_name, fld_type)

# Creating lookup from OID to NN prediction
nn_prob_lookup = {int(oid): float(prob) for oid, prob in zip(nn_oid_test, nn_y_prob)}
nn_pred_lookup = {int(oid): int(pred) for oid, pred in zip(nn_oid_test, nn_y_pred)}

with arcpy.da.UpdateCursor(nn_test_fc, [nn_oid_field_name, "NN_Prob", "NN_Pred"]) as cursor:
    for oid, prob_val, pred_val in cursor:
        new_prob = nn_prob_lookup.get(int(oid), None)
        new_pred = nn_pred_lookup.get(int(oid), None)
        if new_prob is not None and new_pred is not None:
            cursor.updateRow((oid, new_prob, new_pred))

print("Finishing Neural Network training result export")
